# Bayesian Statistics CAT 2


1. Data Preprocessing


In [1]:
import pandas as pd
import numpy as np

In [2]:
# Load the dataset
df=pd.read_excel(r'C:\Users\HP\Desktop\Kepler Courses\Bayesian statistics\Retail_Sales.xlsx')
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\HP\\Desktop\\Kepler Courses\\Bayesian statistics\\Retail_Sales.xlsx'

In [408]:
# Check the structure and data types of the dataset
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   transactions_id  2000 non-null   int64         
 1   sale_date        2000 non-null   datetime64[us]
 2   sale_time        2000 non-null   object        
 3   customer_id      2000 non-null   int64         
 4   gender           2000 non-null   str           
 5   age              1990 non-null   float64       
 6   category         2000 non-null   str           
 7   quantity         1997 non-null   float64       
 8   price_per_unit   1997 non-null   float64       
 9   cogs             1997 non-null   float64       
 10  total_sale       1997 non-null   float64       
dtypes: datetime64[us](1), float64(5), int64(2), object(1), str(2)
memory usage: 172.0+ KB


In [409]:
df.describe()

,transactions_id,sale_date,customer_id,age,quantity,price_per_unit,cogs,total_sale
count,2000.000000,2000,2000.000000,1990.000000,1997.000000,1997.000000,1997.000000,1997.000000
mean,1000.500000,2023-02-16 22:10:33.600000,66.341500,41.343216,2.512769,180.117677,95.023886,456.544817
min,1.000000,2022-01-01 00:00:00,1.000000,18.000000,1.000000,25.000000,6.250000,25.000000
25%,500.750000,2022-09-23 00:00:00,24.000000,29.000000,1.000000,30.000000,13.000000,60.000000
50%,1000.500000,2023-01-12 12:00:00,69.000000,42.000000,3.000000,50.000000,27.500000,150.000000
75%,1500.250000,2023-09-14 00:00:00,102.000000,53.000000,4.000000,300.000000,147.000000,900.000000
max,2000.000000,2023-12-31 00:00:00,155.000000,64.000000,4.000000,500.000000,620.000000,2000.000000
std,577.494589,NaN,44.937185,13.668167,1.132708,189.685225,121.898695,560.101381


In [410]:
# Drop unnecessary columns, that are not needed for the analysis
df.drop(columns=['sale_time', 'cogs'], inplace=True)

In [411]:
# Check for missing values in the dataset
df.isnull().sum()

transactions_id     0
sale_date           0
customer_id         0
gender              0
age                10
category            0
quantity            3
price_per_unit      3
total_sale          3
dtype: int64

In [412]:
# Handle the missing values using fillna() method, 
# First check the skewness of the data to decide whether to use mean or median for imputation but datetime arrays cannot be used to calculate skewness, so we will check the skewness of the numerical columns only
import matplotlib.pyplot as plt
df_skewness = df[['age', 'quantity', 'price_per_unit' ,'total_sale']].skew()
print(df_skewness)



age              -0.042255
quantity         -0.011985
price_per_unit    0.733427
total_sale        1.372853
dtype: float64


In [413]:
# Handle the missing values using fillna() method
df['age'] = df['age'].fillna(df['age'].median())
df['quantity'] = df['quantity'].fillna(df['quantity'].median())
df['price_per_unit'] = df['price_per_unit'].fillna(df['price_per_unit'].median())
df['total_sale'] = df['total_sale'].fillna(df['total_sale'].median())

In [414]:
print("\nMissing values after cleaning:")
df.isnull().sum()


Missing values after cleaning:


transactions_id    0
sale_date          0
customer_id        0
gender             0
age                0
category           0
quantity           0
price_per_unit     0
total_sale         0
dtype: int64

In [415]:
# Convert the data types of the columns to appropriate types
df['sale_date'] = pd.to_datetime(df['sale_date'])
df['age'] = df['age'].astype('int') 
df['quantity'] = df['quantity'].astype('int')
df['year'] = df['sale_date'].dt.year
df['month'] = df['sale_date'].dt.month
df['day'] = df['sale_date'].dt.day

In [416]:
# Using cyclical encoding for the month and day columns to capture the cyclical nature of these features.
# Encode Month
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

In [417]:
# Encode Day of Month
# Note: The divisor is 31, the maximum possible value for a day.
df['day_sin'] = np.sin(2 * np.pi * df['day'] / 31)
df['day_cos'] = np.cos(2 * np.pi * df['day'] / 31)

In [418]:
# Display the first few rows of the cleaned and transformed dataset
df.head()

,transactions_id,sale_date,customer_id,gender,age,category,quantity,price_per_unit,total_sale,year,month,day,month_sin,month_cos,day_sin,day_cos
0,180,2022-11-05,117,Male,41,Clothing,3,300.0,900.0,2022,11,5,-5.000000e-01,0.866025,0.848644,0.528964
1,522,2022-07-09,52,Male,46,Beauty,3,500.0,1500.0,2022,7,9,-5.000000e-01,-0.866025,0.968077,-0.250653
2,559,2022-12-12,5,Female,40,Clothing,4,300.0,1200.0,2022,12,12,-2.449294e-16,1.000000,0.651372,-0.758758
3,1180,2022-01-06,85,Male,41,Clothing,3,300.0,900.0,2022,1,6,5.000000e-01,0.866025,0.937752,0.347305
4,1522,2022-11-14,48,Male,46,Beauty,3,500.0,1500.0,2022,11,14,-5.000000e-01,0.866025,0.299363,-0.954139


In [419]:
# Encode the gender
df['gender_enc'] = df['gender'].apply(lambda val:1 if val=='Male' else 0)

In [420]:
# Remove the unnecessary columns
df = df.drop(columns = ['transactions_id', 'sale_date', 'customer_id', 'month', 'day','gender'])

In [421]:
df = pd.get_dummies(df)

As our second goal will be to classify transactions into value-based categories.

The dataset does not have a categorical target for this purpose. We will create one by binning the `total_sale` column into three categories: 'Low', 'Medium', and 'High'. This allows the business to segment transactions by value.


In [422]:
# Create the target category upfront so the dataframe is clean
buckets = 3
labels = ['Low', 'Medium', 'High']
df['sale_category'] = pd.cut(df['total_sale'], bins=buckets, labels=labels)

In [423]:
df.head()

,age,quantity,price_per_unit,total_sale,year,month_sin,month_cos,day_sin,day_cos,gender_enc,category_Beauty,category_Clothing,category_Electronics,sale_category
0,41,3,300.0,900.0,2022,-5.000000e-01,0.866025,0.848644,0.528964,1,False,True,False,Medium
1,46,3,500.0,1500.0,2022,-5.000000e-01,-0.866025,0.968077,-0.250653,1,True,False,False,High
2,40,4,300.0,1200.0,2022,-2.449294e-16,1.000000,0.651372,-0.758758,0,False,True,False,Medium
3,41,3,300.0,900.0,2022,5.000000e-01,0.866025,0.937752,0.347305,1,False,True,False,Medium
4,46,3,500.0,1500.0,2022,-5.000000e-01,0.866025,0.299363,-0.954139,1,True,False,False,High


In [424]:

# the 3 columns that require scaling now (StandardScaler transforms features so they have a mean of μ = 0 and standard deviation of σ = 1)
features_to_scale = ['age', 'quantity', 'price_per_unit']

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
# Create a transformer that scales only those 3 columns and passes everything else through untouched
preprocessor = ColumnTransformer(transformers=[('num_scaler', StandardScaler(), features_to_scale)],remainder='passthrough')


### 2.1. Prediction with `BayesianRidge`

Our first goal is to predict the `total_sale` amount. This is a regression problem.


In [425]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import BayesianRidge
from sklearn.pipeline import make_pipeline
from sklearn.metrics import *

In [432]:
# Define the features and target
X = df.drop(columns=['total_sale', 'sale_category']).astype(float)
y = df['total_sale'].astype(float)


In [433]:
# Split the data
X_train, X_test, y_train, y_test = train_test_split(X , y, test_size=0.2, random_state=42)

# Create a pipeline with StandardScaler and BayesianRidge
# Use standardscaler with median imputation to handle outliers in the data

pipe = make_pipeline(preprocessor, BayesianRidge())

pipe.fit(X_train, y_train)
prediction, std =pipe.predict(X_test, return_std=True)

In [436]:
# Calculate the evaluation metrics

me  = mean_absolute_error(y_test, prediction)
rmse = root_mean_squared_error(y_test, prediction)
r2 = r2_score(y_test, prediction)

print(f"The mean absolute error is {me}")
print(f"The root mean squared error is {rmse}")
print(f"The r2 score is {r2}")

The mean absolute error is 170.36271208609426
The root mean squared error is 206.14632519752794
The r2 score is 0.8683000078616001


Business Interpretation of the model

This model is a strong asset for the business. It successfully predicts the final sale amount with a high degree of accuracy, explaining 87% of the variance in sales (R² = 0.87). This indicates that the features you are using are highly relevant to predicting sales outcomes.

- Average Prediction Error: On average, the model's prediction for the total_sale is off by about $170 (MAE = 170.36). Whether this is acceptable depends on your profit margins, but it provides a clear number for financial planning and forecasting.
- Risk of Large Errors: The model occasionally makes significantly larger errors. We know this because the Root Mean Squared Error (RMSE = 206.15) is notably higher than the average error. This means that while the model is usually reliable, the business must have a plan for instances where the prediction is off by a much larger amount.
- Uncertainty is Quantified: A key strength of using a Bayesian model is that it doesn't just give a prediction; it also tells you its confidence. The std value you captured is a direct measure of this uncertainty, allowing you to trust some predictions more than others.


### 2.2. Classification with `GaussianNB`


In [437]:
# Define the buckets and labels for classification
buckets = [-np.inf, 500, 1500, np.inf]
labels = ['Low', 'Medium', 'High']
df['sale_category'] = pd.cut(df['total_sale'], bins=buckets, labels=labels)

In [438]:
# Define features (X) and the new target (y)
X_class = df.drop(columns=['sale_category', 'total_sale'])
y_class = df['sale_category']

In [440]:
from sklearn.naive_bayes import GaussianNB
# Split the data
X_train, X_test, y_train, y_test = train_test_split( X_class, y_class, test_size=0.2, stratify = y_class, random_state=42)

pipeline = make_pipeline(preprocessor, GaussianNB())

pipeline.fit(X_train, y_train)
y_prediction = pipeline.predict(X_test)
from sklearn.metrics import classification_report, confusion_matrix

print(f"Unique values in y_test: {np.unique(y_test)}")
print(f"Unique values in y_prediction: {np.unique(y_prediction)}")


print(confusion_matrix(y_test, y_prediction))
print(classification_report(y_test, y_prediction))

Unique values in y_test: ['High' 'Low' 'Medium']
Unique values in y_prediction: ['High' 'Low' 'Medium']
[[ 20   0   0]
 [  0 254  26]
 [  0   4  96]]
              precision    recall  f1-score   support

        High       1.00      1.00      1.00        20
         Low       0.98      0.91      0.94       280
      Medium       0.79      0.96      0.86       100

    accuracy                           0.93       400
   macro avg       0.92      0.96      0.94       400
weighted avg       0.94      0.93      0.93       400



2.3 Key Business takeaways from the Classification Model

- 'High' Sales are Perfectly Identified: The model is now flawless at identifying high-value sales (> 1500). Any business process triggered by this prediction can be fully automated with confidence. This is a major success. <br>
- 'Low' Sales are Reliably Identified: The model is very strong here (98% precision). When the model says a sale is 'Low', you can trust it. <br>
- 'Medium' Sales are the New Challenge: The model's main weakness is now the 'Medium' category. <br>
  It finds almost all 'Medium' sales (96% recall).<br>
  However, its predictions are only correct 79% of the time because it incorrectly pulls some 'Low' sales into this category.


### 3. Bayesian Statistics: Strengths and Limitations

Strengths:

1. Quantifies Uncertainty: The BayesianRidge model doesn't just give a sales prediction; it also tells you its level of confidence (the std value). This is its single greatest strength. For business, this means you can create rules to act only on high-confidence predictions, which is critical for managing risk and resources.
2. Provides a Strong Baseline: Both models deliver solid performance (R² of 0.87 for regression, 93% accuracy for classification) with high efficiency. They are excellent starting points for any prediction task.
3. It's Robust: The underlying math helps prevent the model from overfitting to the training data, leading to more stable and reliable performance on new, unseen data.

Limitations:

1. The "Naive" Assumption is a Weakness: The GaussianNB classifier assumes all your business features are independent, which is rarely true. This simplifying assumption is the likely reason it still confuses 'Low' and 'Medium' sales, it can't capture the complex interactions between features.
2. Raw Performance Can Be Beaten: While efficient, Bayesian models may not achieve the absolute highest accuracy compared to more complex, computationally expensive models like XGBoost or Neural Networks, which are better at learning complex, non-linear patterns.
3. Uncertainty Can Be Hard to Explain: Translating a "prediction's standard deviation" into a simple business rule requires careful thought and can be less intuitive to stakeholders than a simple "right or wrong" accuracy metric.
